# SEA Concepts — Abrupt Drift

Worked single-dataset example for the benchmark. Establishes that the injected
drift is real, then compares all adaptation strategies against it.

Run order: validate drift → baseline → all strategies → plots. Everything is
logged to MLflow (`file:./mlruns`); view with `mlflow ui` from the repo root.


In [ ]:
import sys; sys.path.insert(0, "../src")
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import pandas as pd

import streams, pipeline
from experiments import run_grid, summarise, best_per_dataset

plt.rcParams["figure.figsize"] = (11, 4.5)

## 1. Build the stream

SEA variants differ only in the threshold on `att1 + att2`
(`{0: 8, 1: 9, 2: 7, 3: 9.5}`). We use **(3, 2)** — a 9.5 → 7 shift, the widest
gap available.

This choice is load-bearing. The narrower (0, 2) pairing shifts the boundary by
only 1.0 across a sum spanning 0–20, relabelling ~7% of the space; a Hoeffding
Tree absorbs that with **no measurable accuracy drop**, which makes recovery
time undefined and every strategy comparison meaningless. Cell 3 exists to
catch exactly that failure.

In [ ]:
samples, drift_points = streams.sea(n_per_concept=5000, variants=(3, 2), seed=42)
print(f"{len(samples)} samples, true drift at {drift_points}")
samples[0]

## 2. Validate the drift is real

Before comparing anything, confirm a static learner actually degrades at the
drift point. If accuracy does not visibly drop here, the stream has no usable
drift and no downstream result means anything.

In [ ]:
baseline = pipeline.run_strategy(
    samples, "baseline", "sea_abrupt", drift_points, log_to_mlflow=False
)

curve = baseline["curve"]
pre  = [a for i, a in curve if i <= drift_points[0]][-1]
post = [a for i, a in curve if i >  drift_points[0]][0]
print(f"windowed accuracy before drift: {pre:.3f}")
print(f"windowed accuracy after  drift: {post:.3f}")
print(f"drop: {pre - post:+.3f}")
assert pre - post > 0.05, "No measurable drift — fix the stream before continuing."
print("\nDrift confirmed.")

In [ ]:
xs, ys = zip(*curve)
plt.plot(xs, ys, lw=1.4, color="#444")
for d in drift_points:
    plt.axvline(d, color="crimson", ls="--", label="true drift")
plt.title("Baseline (no adaptation) — accuracy collapses at the drift point")
plt.xlabel("sample"); plt.ylabel("windowed accuracy"); plt.legend()
plt.ylim(0.5, 1.0); plt.grid(alpha=.3); plt.show()

## 3. Compare all strategies

Each strategy runs on the same stream. `detector_incremental` is a deliberate
control — ADWIN fires but nothing is reset, so it isolates how much adaptation
comes from the detector versus the incremental learner's own updating.

In [ ]:
results = {}
for name in pipeline.STRATEGIES:
    results[name] = pipeline.run_strategy(
        samples, name, "sea_abrupt", drift_points, seed=42, log_to_mlflow=True
    )
    r = results[name]
    print(f"{name:24s} acc={r['final_accuracy']:.4f} "
          f"f1={r['final_macro_f1']:.4f} recovery={r['recoveries']} "
          f"detections={r['n_drift_detections']:3d} "
          f"{r['runtime_sec']:5.1f}s {r['peak_memory_mb']:.1f}MB")

In [ ]:
fig, ax = plt.subplots()
for name, r in results.items():
    xs, ys = zip(*r["curve"])
    ax.plot(xs, ys, lw=1.3, label=name, alpha=.85)
for d in drift_points:
    ax.axvline(d, color="crimson", ls="--", lw=1)
ax.set_title("Adaptation strategies through an abrupt drift")
ax.set_xlabel("sample"); ax.set_ylabel("windowed accuracy")
ax.set_ylim(0.5, 1.0); ax.grid(alpha=.3)
ax.legend(loc="lower right", fontsize=9); plt.show()

### Accuracy vs. compute

The core tradeoff for the paper: ARF buys accuracy with roughly an order of
magnitude more runtime and memory. Whether that is worth it is a
budget question, and is the finding this plot supports.

In [ ]:
cost = pd.DataFrame([
    {"strategy": k, "accuracy": r["final_accuracy"],
     "runtime_sec": r["runtime_sec"], "memory_mb": r["peak_memory_mb"]}
    for k, r in results.items()
])

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(cost.runtime_sec, cost.accuracy, s=cost.memory_mb * 40 + 60,
           alpha=.65, color="#2b6cb0")
for _, row in cost.iterrows():
    ax.annotate(row.strategy, (row.runtime_sec, row.accuracy),
                textcoords="offset points", xytext=(8, 5), fontsize=9)
ax.set_xlabel("runtime (s)  →  more expensive")
ax.set_ylabel("final accuracy")
ax.set_title("Accuracy vs. compute cost (marker size = peak memory)")
ax.grid(alpha=.3); plt.show()
cost.round(4)

## 4. Full grid

All datasets × strategies × 3 seeds. Multiple seeds matter: one run on one
seeded stream is n=1 and cannot support a claim about which strategy is best.

Takes a few minutes — ARF dominates the runtime.

In [ ]:
df = run_grid(log_to_mlflow=True)
df.to_csv("../results/raw_runs.csv", index=False)

agg = summarise(df)
agg.to_csv("../results/summary.csv", index=False)
agg.round(4)

### Best strategy per dataset

`separable` is the column that matters. When the winner's margin over the
runner-up is smaller than its own standard deviation across seeds, the two are
**not** statistically distinguishable and must not be reported as a ranking —
report them as tied, or add seeds.

In [ ]:
best = best_per_dataset(agg)
best.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
pivot = agg.pivot(index="dataset", columns="strategy", values="acc_mean")
errs  = agg.pivot(index="dataset", columns="strategy", values="acc_std")
pivot.plot.bar(ax=ax, yerr=errs, capsize=3, alpha=.9, rot=0)
ax.set_ylabel("mean final accuracy (±1 std over seeds)")
ax.set_title("Strategy performance by dataset")
ax.set_ylim(0.5, 1.0); ax.grid(alpha=.3, axis="y")
ax.legend(fontsize=8, ncol=3); plt.show()

## 5. Real-world streams

Elec2 and Airlines differ from the synthetic streams in two ways that change how
results must be read:

1. **No ground-truth drift index.** Nobody knows when the concept actually
   changed, so recovery time is not computed. Comparisons rest on accuracy,
   macro-F1, runtime/memory, and how often the detector fired.
2. **Fixed rows in a fixed order.** The seed changes nothing, so the
   deterministic strategies run once; only ARF is repeated across seeds. A
   `separable` of `None` below means a single run cannot answer the question —
   it is not the same as a measured tie.

Airlines is capped at the first 100,000 rows in temporal order. Requires
`python scripts/download_data.py` first.

In [ ]:
import streams

# Airlines parses 100k rows from parquet - takes a few seconds.
elec_samples, _ = streams.elec2()
air_samples, _   = streams.airlines(n_samples=100_000)
print(f"elec2:    {len(elec_samples):,} samples, {len(elec_samples[0][0])} features")
print(f"airlines: {len(air_samples):,} samples, {len(air_samples[0][0])} features")
print("\nairlines has categorical features - River trees split on them natively:")
air_samples[0][0]

### Elec2

Well-studied benchmark with recurring drift driven by demand cycles.

In [ ]:
# ~2.5 min: ARF alone is ~2 min on Elec2's 45k rows, the rest are seconds.
elec_results = {}
for name in pipeline.STRATEGIES:
    elec_results[name] = pipeline.run_strategy(
        elec_samples, name, "elec2", (), log_to_mlflow=False
    )
    r = elec_results[name]
    print(f"{name:24s} acc={r['final_accuracy']:.4f} f1={r['final_macro_f1']:.4f} "
          f"detections={r['n_drift_detections']:3d} {r['runtime_sec']:6.1f}s", flush=True)

In [ ]:
fig, ax = plt.subplots()
for name, r in elec_results.items():
    xs, ys = zip(*r["curve"])
    ax.plot(xs, ys, lw=1.1, label=name, alpha=.85)
ax.set_title("Elec2 — no known drift points, so no reference lines")
ax.set_xlabel("sample"); ax.set_ylabel("windowed accuracy")
ax.grid(alpha=.3); ax.legend(loc="lower right", fontsize=9); plt.show()

### Detector firing as a drift proxy

With no ground truth, ADWIN's firing points are the closest thing to a drift
signal these streams offer. Read them as *where the error rate shifted*, not as
verified concept changes — a detector can fire on noise, and can miss a real
change that does not move the error rate.

In [ ]:
det = elec_results["detector_full_retrain"]
fires = det["detected_drifts"]
print(f"ADWIN fired {len(fires)} times on Elec2")

fig, ax = plt.subplots()
xs, ys = zip(*det["curve"])
ax.plot(xs, ys, lw=1.2, color="#444", label="detector_full_retrain")
for f in fires:
    ax.axvline(f, color="crimson", alpha=.25, lw=.8)
ax.set_title(f"Elec2 — ADWIN firing points ({len(fires)} detections)")
ax.set_xlabel("sample"); ax.set_ylabel("windowed accuracy")
ax.grid(alpha=.3); ax.legend(fontsize=9); plt.show()

### Airlines

The hardest stream in the benchmark: real operational data, implicit drift, and
mixed categorical/numeric features. Accuracy here sits far below the synthetic
streams — that is the task, not a bug.

ARF costs ~5 minutes per run on 100k rows, so the cell below runs only the four
cheap strategies (~1 min total). ARF's numbers are read from the grid results
in section 6 rather than recomputed here.

In [ ]:
# ~1 min: the four deterministic strategies. ARF is excluded by cost -
# see section 6 for its numbers from the full grid.
air_results = {}
for name in ["baseline", "blind_periodic", "detector_full_retrain",
             "detector_incremental"]:
    air_results[name] = pipeline.run_strategy(
        air_samples, name, "airlines", (), log_to_mlflow=False
    )
    r = air_results[name]
    print(f"{name:24s} acc={r['final_accuracy']:.4f} f1={r['final_macro_f1']:.4f} "
          f"detections={r['n_drift_detections']:3d} {r['runtime_sec']:6.1f}s", flush=True)

In [ ]:
fig, ax = plt.subplots()
for name, r in air_results.items():
    xs, ys = zip(*r["curve"])
    ax.plot(xs, ys, lw=1.1, label=name, alpha=.85)
ax.set_title("Airlines — first 100k rows, temporal order preserved")
ax.set_xlabel("sample"); ax.set_ylabel("windowed accuracy")
ax.grid(alpha=.3); ax.legend(loc="lower right", fontsize=9); plt.show()

## 6. Final comparison

The full picture across all five datasets. Read `separable` before quoting any
winner: `True` means the margin exceeds the winner's own spread, `False` means
the top two are statistically indistinguishable, and `None` means the run was
not replicated and the question stays open.

In [ ]:
df  = pd.read_csv("../results/raw_runs.csv")
agg = summarise(df)
best = best_per_dataset(agg)
best.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))
pivot = agg.pivot(index="dataset", columns="strategy", values="acc_mean")
errs  = agg.pivot(index="dataset", columns="strategy", values="acc_std")
pivot.plot.bar(ax=ax, yerr=errs, capsize=3, alpha=.9, rot=0)
ax.set_ylabel("mean final accuracy")
ax.set_title("All datasets — synthetic and real")
ax.grid(alpha=.3, axis="y"); ax.legend(fontsize=8, ncol=3); plt.show()

In [ ]:
# Accuracy vs. compute across every dataset: does ARF's cost buy anything?
fig, ax = plt.subplots(figsize=(8, 5.5))
for ds, grp in agg.groupby("dataset"):
    ax.plot(grp.runtime_mean, grp.acc_mean, "o", ms=9, alpha=.7, label=ds)
    for _, row in grp.iterrows():
        if row.strategy == "arf":
            ax.annotate("ARF", (row.runtime_mean, row.acc_mean),
                        textcoords="offset points", xytext=(7, 3), fontsize=8)
ax.set_xscale("log")
ax.set_xlabel("mean runtime (s, log scale)"); ax.set_ylabel("mean accuracy")
ax.set_title("Accuracy vs. compute — ARF is the rightmost point in every stream")
ax.grid(alpha=.3); ax.legend(fontsize=9); plt.show()

## Next

- Base-learner sweep (Hoeffding Tree / Naive Bayes / Logistic Regression) via
  `model_fn` in `STRATEGIES`.
- ADWIN `delta` sensitivity: detection sensitivity trades false alarms against
  detection delay.
- Add DDM alongside ADWIN.
- **Fix `recovery_mean` on `sea_gradual`** — it currently bottoms out at one
  `eval_every` window, reading 100 for every strategy including the baseline.
  That is the metric's resolution limit, not a real tie; tighten `eval_every`
  or widen the transition before reporting it.
- Airlines currently uses the first 100k of 539k rows; rerun at full size if
  the compute budget allows.
- `mlflow ui --backend-store-uri sqlite:///mlflow.db` to browse every run.
